# StreamLoop — tuning the churn model

This notebook demonstrates the complete leakage-safe workflow: direct loading from the official URL, a held-out stratified test split, learned preprocessing inside a pipeline, a default baseline, randomized exploration, grid refinement, fold-stability inspection, and one final test evaluation. F2 is primary because missing a real churner costs more than an unnecessary retention offer, while precision still prevents an all-positive classifier.

In [1]:
from pathlib import Path
import pandas as pd
from streamloop.training import DATA_URL, load_dataset, make_pipeline, run_training

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
features, target = load_dataset(DATA_URL)
print(DATA_URL)
print(f'rows={len(features):,}; churn rate={target.mean():.2%}')
features.head()

https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
rows=7,043; churn rate=26.54%


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [2]:
pipeline = make_pipeline()
print(pipeline)
print('Default classifier parameters:', {
    key: pipeline.named_steps['classifier'].get_params()[key]
    for key in ['C', 'class_weight', 'solver']
})

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x114f7c980>),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='

In [3]:
result = run_training(ROOT)
pd.DataFrame({
    'baseline': result['baseline_test'],
    'tuned': result['final_test'],
}).T

,recall,precision,f1,f2,accuracy,roc_auc
baseline,0.558824,0.657233,0.604046,0.576075,0.805536,0.841861
tuned,0.783422,0.506045,0.614900,0.706024,0.739532,0.841091


In [4]:
pd.DataFrame(result['grid_search']['top_candidates'])[['rank', 'mean_cv_f2', 'std_cv_f2', 'params']]

,rank,mean_cv_f2,std_cv_f2,params
0,1,0.722846,0.029905,"{'classifier__C': 3.1262673090251236, 'classif..."
1,2,0.722072,0.028975,"{'classifier__C': 1.5631336545125618, 'classif..."
2,3,0.721899,0.030141,"{'classifier__C': 4.689400963537685, 'classifi..."
3,4,0.721700,0.028703,"{'classifier__C': 0.5210445515041873, 'classif..."
4,5,0.721305,0.030422,"{'classifier__C': 7.034101445306527, 'classifi..."


## Interpretation

The tuned model intentionally trades some precision and raw accuracy for higher churn recall, while F2 guards against a degenerate all-positive result. This matches StreamLoop's stated cost asymmetry. Fold standard deviation is reviewed alongside mean F2, and the final parameter set is accepted because its stability is comparable to nearby candidates. The detailed decision and exact parameters are generated in `tuning_report.md`.